# Week 04 Lab Work

**Dataset:** Adult / Census Income (UCI Machine Learning Repository) - `adult.data`

1. Take a dataset from the internet (used for all future labs).
2. Calculate the dissimilarity or similarity of a data matrix.
3. Calculate the dissimilarity of a mix type of attributes.

The Adult dataset is used because it contains all four attribute types needed for Task 3:
numeric, nominal, binary and ordinal.

## 1. Load the dataset

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import urllib.request

cols = ["age", "workclass", "fnlwgt", "education", "education-num",
        "marital-status", "occupation", "relationship", "race", "sex",
        "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]

data_filename = "adult.data"
target_path = None

# Search dynamically for adult.data
for root, dirs, files in os.walk("."):
    if data_filename in files:
        target_path = os.path.join(root, data_filename)
        break

# If not found locally, download it directly from the UCI repository
if target_path is None:
    print(f"'{data_filename}' not found in the workspace. Downloading it dynamically...")
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
    try:
        urllib.request.urlretrieve(url, data_filename)
        target_path = data_filename
        print("Download complete!")
    except Exception as e:
        raise FileNotFoundError(
            f"Could not download the file automatically. Please manually upload '{data_filename}' "
            "to the Colab file explorer panel on the left."
        ) from e

print(f"Loading dataset from: {target_path}")
df = pd.read_csv(target_path, header=None, names=cols,
                 skipinitialspace=True, na_values="?")
df = df.dropna().reset_index(drop=True)
df.shape

'adult.data' not found in the workspace. Downloading it dynamically...
Download complete!
Loading dataset from: adult.data


(30162, 15)

In [ ]:
df.head()

### Attribute types

| Type | Attributes |
|---|---|
| Numeric (ratio/interval) | `age`, `capital-gain`, `capital-loss`, `hours-per-week` |
| Nominal | `workclass`, `occupation`, `marital-status`, `race`, `native-country` |
| Binary (asymmetric) | `sex` (Male=1), `income` (>50K=1) |
| Ordinal | `education` ranked by `education-num` (1 = Preschool ... 16 = Doctorate) |

### Take a small sample

A distance matrix of the full 30,162 rows would be 30,162 x 30,162.
A sample of 10 objects is used so every matrix can be printed.

In [ ]:
S = df.sample(10, random_state=42).reset_index(drop=True)
S.index = [f"o{i+1}" for i in range(len(S))]
S[["age", "hours-per-week", "capital-gain", "education", "education-num",
   "workclass", "occupation", "race", "sex", "income"]]

## 2. Dissimilarity / similarity of a data matrix

### 2.1 Numeric attributes

Values are min-max normalised to [0, 1] first, so that `capital-gain` (range ~100,000)
does not drown out `age` (range ~70).

$$z = \frac{x - min}{max - min}$$

In [ ]:
numeric = ["age", "capital-gain", "capital-loss", "hours-per-week"]

lo, hi = df[numeric].min(), df[numeric].max()
rng = (hi - lo).replace(0, 1)
Z = (S[numeric] - lo) / rng
Z.round(3)

Minkowski distance of order $p$:

$$d(i,j) = \left(\sum_{f=1}^{n} |x_{if} - x_{jf}|^{p}\right)^{1/p}$$

$p = 1$ gives Manhattan distance, $p = 2$ gives Euclidean distance.

In [ ]:
def minkowski_matrix(X, p):
    diff = np.abs(X.values[:, None, :] - X.values[None, :, :]) ** p
    return pd.DataFrame(diff.sum(axis=2) ** (1 / p), index=X.index, columns=X.index)

manhattan = minkowski_matrix(Z, 1)
euclidean = minkowski_matrix(Z, 2)

print("Manhattan (p = 1)")
display(manhattan.round(3))
print("Euclidean (p = 2)")
display(euclidean.round(3))

### 2.2 Nominal attributes

Simple matching: count how many attributes disagree.

$$d(i,j) = \frac{p - m}{p}$$

where $p$ is the number of attributes and $m$ the number of matches.

In [ ]:
nominal = ["workclass", "occupation", "marital-status", "race", "native-country"]

V = S[nominal].to_numpy(dtype=object)
p = len(nominal)
mismatch = (V[:, None, :] != V[None, :, :]).sum(axis=2)
nominal_d = pd.DataFrame(mismatch / p, index=S.index, columns=S.index)
nominal_d.round(3)

### 2.3 Binary attributes (asymmetric)

`sex = Male` and `income > 50K` are treated as the rare/positive outcome (1).
For asymmetric binary attributes the 0-0 match carries no information, so the
Jaccard coefficient is used:

$$d(i,j) = \frac{b + c}{a + b + c}$$

where $a$ = attributes that are 1 in both objects, $b$ and $c$ = the mismatches.

In [ ]:
B = pd.DataFrame({"sex_male": (S["sex"] == "Male").astype(int),
                  "income_gt50k": (S["income"] == ">50K").astype(int)},
                 index=S.index)
display(B)

X = B.values
a = ((X[:, None, :] == 1) & (X[None, :, :] == 1)).sum(axis=2)
b_c = (X[:, None, :] != X[None, :, :]).sum(axis=2)
denom = np.where(a + b_c == 0, 1, a + b_c)
binary_d = pd.DataFrame(b_c / denom, index=S.index, columns=S.index)
binary_d.round(3)

### 2.4 Ordinal attribute

`education` has a natural order. Its rank $r \in \{1 \dots M\}$ is already stored in
`education-num` ($M = 16$). The rank is mapped to [0, 1] and then treated as numeric:

$$z = \frac{r - 1}{M - 1}, \qquad d(i,j) = |z_i - z_j|$$

In [ ]:
M = df["education-num"].max()
z_edu = (S["education-num"] - 1) / (M - 1)
display(pd.DataFrame({"education": S["education"], "rank": S["education-num"],
                      "z": z_edu.round(3)}))

ordinal_d = pd.DataFrame(np.abs(z_edu.values[:, None] - z_edu.values[None, :]),
                         index=S.index, columns=S.index)
ordinal_d.round(3)

### 2.5 Similarity

Cosine similarity on the normalised numeric matrix. It measures the angle between two
object vectors, so it ranges from 0 (unrelated) to 1 (identical direction).

$$sim(i,j) = \frac{x_i \cdot x_j}{\|x_i\| \, \|x_j\|}$$

In [ ]:
norms = np.linalg.norm(Z.values, axis=1)
norms = np.where(norms == 0, 1, norms)
cosine_sim = pd.DataFrame((Z.values @ Z.values.T) / np.outer(norms, norms),
                          index=S.index, columns=S.index)
cosine_sim.round(3)

## 3. Dissimilarity of mixed type attributes

All four types are combined into one matrix with the weighted formula:

$$d(i,j) = \frac{\sum_{f=1}^{n} \delta_{ij}^{(f)} \, d_{ij}^{(f)}}{\sum_{f=1}^{n} \delta_{ij}^{(f)}}$$

Per attribute $f$:

| Type | $d_{ij}^{(f)}$ | $\delta_{ij}^{(f)}$ |
|---|---|---|
| Numeric | $\dfrac{\lvert x_{if} - x_{jf}\rvert}{max_f - min_f}$ | 1 |
| Nominal | 0 if equal, else 1 | 1 |
| Ordinal | $\lvert z_{if} - z_{jf}\rvert$ after rank normalisation | 1 |
| Asymmetric binary | 0 if equal, else 1 | 0 if both are 0, else 1 |

Every $d_{ij}^{(f)}$ is already in [0, 1], so the result is also in [0, 1].

In [ ]:
def mixed_dissimilarity(S):
    n = len(S)
    contrib = np.zeros((n, n))
    weight = np.zeros((n, n))

    for f in numeric:
        x = S[f].values.astype(float)
        r = hi[f] - lo[f] or 1
        contrib += np.abs(x[:, None] - x[None, :]) / r
        weight += 1

    for f in nominal:
        x = S[f].to_numpy(dtype=object)
        contrib += (x[:, None] != x[None, :]).astype(float)
        weight += 1

    x = z_edu.values
    contrib += np.abs(x[:, None] - x[None, :])
    weight += 1

    for f in B.columns:
        x = B[f].values
        delta = ~((x[:, None] == 0) & (x[None, :] == 0))
        contrib += (x[:, None] != x[None, :]) * delta
        weight += delta

    return pd.DataFrame(contrib / weight, index=S.index, columns=S.index)

mixed_d = mixed_dissimilarity(S)
mixed_d.round(3)

### Heatmap of the mixed-type dissimilarity matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(mixed_d.values, cmap="viridis")
ax.set_xticks(range(len(S)), S.index)
ax.set_yticks(range(len(S)), S.index)
ax.set_title("Mixed-type dissimilarity")
for i in range(len(S)):
    for j in range(len(S)):
        ax.text(j, i, f"{mixed_d.values[i, j]:.2f}", ha="center", va="center",
                color="white" if mixed_d.values[i, j] < 0.5 else "black", fontsize=7)
fig.colorbar(im, label="dissimilarity")
plt.tight_layout()
plt.show()

### Most similar and most different pair

In [ ]:
d = mixed_d.values.copy()
np.fill_diagonal(d, np.nan)
i, j = np.unravel_index(np.nanargmin(d), d.shape)
k, l = np.unravel_index(np.nanargmax(d), d.shape)

print(f"most similar : {S.index[i]} and {S.index[j]}  d = {d[i, j]:.3f}")
print(f"most different: {S.index[k]} and {S.index[l]}  d = {d[k, l]:.3f}")

cmp = ["age", "hours-per-week", "education", "workclass", "occupation",
       "marital-status", "race", "sex", "income"]
display(S.loc[[S.index[i], S.index[j]], cmp].T.rename(columns={S.index[i]: "A", S.index[j]: "B"}))
display(S.loc[[S.index[k], S.index[l]], cmp].T.rename(columns={S.index[k]: "A", S.index[l]: "B"}))

## Summary

- The dataset for all future labs is **Adult / Census Income (UCI)**, 30,162 complete records.
- Task 2: dissimilarity matrices were built for numeric (Manhattan, Euclidean),
  nominal (simple matching), asymmetric binary (Jaccard) and ordinal (rank normalisation)
  attributes, plus a cosine similarity matrix.
- Task 3: all four types were merged into a single dissimilarity matrix with the weighted
  formula, giving values in [0, 1].